In [1]:
import pandas as pd
import numpy as np
from scipy.stats import mannwhitneyu, chi2_contingency
from statsmodels.stats.multitest import fdrcorrection





In [2]:
# ------------------------------------------------------------
# 1. Load your CSV file (change the path to your actual file)
# ------------------------------------------------------------
df = pd.read_csv("GallStone_dataset-uci.csv")   # <-- replace with your file path

# ------------------------------------------------------------
# 2. Define columns
# ------------------------------------------------------------
target = "Gallstone Status"
# List all feature columns (exclude the target)
feature_cols = [col for col in df.columns if col != target]

In [3]:

# ------------------------------------------------------------
# 3. Helper functions to decide test type and compute p-value
# ------------------------------------------------------------
def is_categorical(series, max_unique=10):
    """Heuristic: if number of unique values <= max_unique and dtype is object/int, treat as categorical."""
    if series.dtype == 'object' or series.dtype.name == 'category':
        return True
    if series.nunique() <= max_unique:
        return True
    return False

# ------------------------------------------------------------
# (A) Function to compute mean ± standard deviation for numeric features in both groups
# ------------------------------------------------------------
def compute_group_stats(df, target, features):
    """
    For each numeric feature, compute mean ± std for group 0 and group 1 (based on target column).
    Returns a DataFrame with columns: Feature, Group0_mean_std, Group1_mean_std
    """
    group0 = df[df[target] == df[target].unique()[0]]
    group1 = df[df[target] == df[target].unique()[1]]
    
    stats = []
    for col in features:
        # Only numeric columns
        if pd.api.types.is_numeric_dtype(df[col]):
            m0, s0 = group0[col].mean(), group0[col].std()
            m1, s1 = group1[col].mean(), group1[col].std()
            stats.append({
                'Feature': col,
                'Group0_mean_std': f"{m0:.2f} ± {s0:.2f}",
                'Group1_mean_std': f"{m1:.2f} ± {s1:.2f}"
            })
        else:
            # For non-numeric, you could add counts or skip
            stats.append({
                'Feature': col,
                'Group0_mean_std': "Non-numeric",
                'Group1_mean_std': "Non-numeric"
            })
    return pd.DataFrame(stats)

def compute_p_value(feature, target, data):
    """Return p-value comparing two groups defined by target (binary)."""
    groups = data[feature].groupby(data[target])
    # Get the two groups (should be 0 and 1 or similar)
    unique_vals = data[target].unique()
    if len(unique_vals) != 2:
        raise ValueError(f"Target {target} must be binary. Found {unique_vals}")
    group0 = groups.get_group(unique_vals[0]).dropna()
    group1 = groups.get_group(unique_vals[1]).dropna()

    # Decide if categorical
    if is_categorical(data[feature]):
        # Chi-square test for independence
        contingency = pd.crosstab(data[target], data[feature])
        # Use chi2_contingency with correction for 2x2 tables
        chi2, p, dof, expected = chi2_contingency(contingency, correction=True)
        return p
    else:
        # Continuous: Mann-Whitney U test (non-parametric alternative to t-test)
        # Use alternative='two-sided'
        u_stat, p = mannwhitneyu(group0, group1, alternative='two-sided')
        return p



In [4]:
# ------------------------------------------------------------
# (B) Compute mean±std for both groups
# ------------------------------------------------------------
group_stats_df = compute_group_stats(df, target, feature_cols)

print("\n=== Mean ± Std for each group (numeric features only) ===")
print(group_stats_df)

group_stats_df.to_csv("group_mean_std_results.csv", index=False)



=== Mean ± Std for each group (numeric features only) ===
                                           Feature  Group0_mean_std  \
0                                              Age    47.63 ± 12.97   
1                                           Gender      0.42 ± 0.49   
2                                      Comorbidity      0.36 ± 0.57   
3                    Coronary Artery Disease (CAD)      0.06 ± 0.23   
4                                   Hypothyroidism      0.04 ± 0.19   
5                                   Hyperlipidemia      0.00 ± 0.00   
6                           Diabetes Mellitus (DM)      0.10 ± 0.30   
7                                           Height    168.23 ± 9.77   
8                                           Weight    79.81 ± 15.98   
9                            Body Mass Index (BMI)     28.24 ± 5.29   
10                          Total Body Water (TBW)     41.46 ± 7.81   
11                       Extracellular Water (ECW)     17.63 ± 2.94   
12                

In [5]:
# ------------------------------------------------------------
# 4. Compute p-values for all features
# ------------------------------------------------------------
p_values = {}
for col in feature_cols:
    # Drop missing values for this feature (listwise per test)
    temp_df = df[[target, col]].dropna()
    if temp_df[col].nunique() < 2:
        p_values[col] = np.nan   # constant feature -> no test
        continue
    try:
        p = compute_p_value(col, target, temp_df)
        p_values[col] = p
    except Exception as e:
        print(f"Error on {col}: {e}")
        p_values[col] = np.nan

# ------------------------------------------------------------
# 5. FDR correction (Benjamini-Hochberg)
# ------------------------------------------------------------
# Remove NaNs before correction
valid_features = [f for f, p in p_values.items() if not np.isnan(p)]
valid_pvals = [p_values[f] for f in valid_features]

if len(valid_pvals) > 0:
    reject, fdr_corrected = fdrcorrection(valid_pvals, alpha=0.05, method='indep')
    # Create result DataFrame
    results = pd.DataFrame({
        'Feature': valid_features,
        'p_value': valid_pvals,
        'FDR': fdr_corrected,
        'Reject_H0': reject
    })
else:
    results = pd.DataFrame(columns=['Feature', 'p_value', 'FDR', 'Reject_H0'])

# ------------------------------------------------------------
# 6. Display results sorted by p-value
# ------------------------------------------------------------
results = results.sort_values('p_value').reset_index(drop=True)
print(results)

# Optionally save to CSV
results.to_csv("pvalues_fdr_results.csv", index=False)

                                           Feature       p_value  \
0                         C-Reactive Protein (CRP)  1.764035e-20   
1                                        Vitamin D  1.363600e-11   
2                   Aspartat Aminotransferaz (AST)  2.379097e-07   
3                               Lean Mass (LM) (%)  4.840835e-05   
4                  Total Body Fat Ratio (TBFR) (%)  4.941677e-05   
5                                   Bone Mass (BM)  9.394115e-05   
6                   Hepatic Fat Accumulation (HFA)  1.364365e-04   
7                          Total Fat Content (TFC)  5.822191e-04   
8                                 Hemoglobin (HGB)  6.265234e-04   
9                        Extracellular Water (ECW)  1.312565e-03   
10                  High Density Lipoprotein (HDL)  1.812496e-03   
11                         Visceral Fat Area (VFA)  5.228238e-03   
12                                          Gender  8.556051e-03   
13                                  Hyperlipidem